In [21]:
import pandas as pd
import numpy as np

from datasets import Dataset
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from transformers import TrainingArguments
from transformers import Trainer

from sklearn.metrics import accuracy_score,f1_score
from sklearn.utils.class_weight import compute_class_weight
import torch
import torch.nn as nn

In [22]:
import re
import emoji

slang_dict = {
    "ga": "tidak",
    "gak": "tidak",
    "gk": "tidak",
    "nggak": "tidak",
    "tp": "tapi",
    "jd": "jadi",
    "bgt": "banget",
    "yg": "yang",
    "krn": "karena"
}

def normalize_slang(text):
    words = text.split()
    return " ".join([slang_dict.get(w, w) for w in words])


def preprocess_text(text):
    text = str(text)

    # 1. Lowercase
    text = text.lower()

    # 2. Remove URL
    text = re.sub(r'http\S+|www\S+', ' ', text)

    # 3. Remove mention
    text = re.sub(r'@\w+', ' ', text)

    # 4. Remove hashtag symbol only (kata tetap)
    text = re.sub(r'#', '', text)

    # 5. Remove emoji
    text = emoji.replace_emoji(text, replace='')

    # 6. Slang normalization ringan
    text = normalize_slang(text)

    # 7. Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text


In [23]:
train_df = pd.read_csv("data/train.csv")
test_df = pd.read_csv("data/test.csv")

In [24]:
train_df["label"] = train_df["label"].astype(int)
test_df["label"] = test_df["label"].astype(int)

In [25]:
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

In [26]:
print(train_dataset.column_names)
print(train_dataset[0])

['comment', 'label']
{'comment': 'berbahaya??', 'label': 0}


In [27]:
model_name = "indobenchmark/indobert-base-p2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    use_safetensors=True
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 451.22it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p2
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [28]:
def tokenize(batch):
    texts = batch["comment"]
    
    # pastikan semua isi list adalah string
    texts = [str(t) for t in texts]

    return tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=128
    )

In [29]:
train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/1973 [00:00<?, ? examples/s]

Map: 100%|██████████| 494/494 [00:00<00:00, 11283.91 examples/s]


In [30]:
def compute_metrics(eval_pred):

    logits,labels = eval_pred

    preds = np.argmax(logits, axis=1)

    acc = accuracy_score(labels,preds)

    f1 = f1_score(labels,preds,average="macro")

    return {"accuracy":acc,"f1":f1}

In [31]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="models/sentiment_indoroberta",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",   # ganti dari evaluation_strategy
    logging_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01
)

In [32]:
# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=train_dataset,
#     eval_dataset=test_dataset,
#     processing_class=tokenizer,
#     compute_metrics=compute_metrics
# )

In [33]:
labels = train_df["label"].values

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels),
    y=labels
)

class_weights = torch.tensor(class_weights, dtype=torch.float)

print("Class Weights:", class_weights)

Class Weights: tensor([1.2551, 0.5704, 2.2218])


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [38]:
class_weights = class_weights.to(device)

NameError: name 'device' is not defined

In [34]:
import torch.nn as nn
from transformers import Trainer

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        loss_fct = nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

In [35]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,  # ✅ benar
    compute_metrics=compute_metrics
)

In [36]:
print(train_dataset.column_names)
print(train_dataset[0])

['comment', 'label', 'input_ids', 'token_type_ids', 'attention_mask']
{'comment': 'berbahaya??', 'label': 0, 'input_ids': [2, 4199, 30477, 30477, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [37]:
trainer.train()

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu! (when checking argument for argument weight in method wrapper_CUDA_nll_loss_forward)

In [ ]:
trainer.save_model("models/sentiment_indobert")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.62it/s]
